In [1]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Softmax fusion temperature(tau) 그리드 서치 결과를 비교한다.
# 참고: run_cue_tau_1.sh / run_cue_tau_2.sh
cue_base = '/data/rlawlsgurjh/work/gitlab/CUE/cue/results'

datasets = ['DIS', 'INH']
targets = ['PI', 'Ssel', 'WS2', 'RS2']
metrics_of_interest = ['rmse', 'r2', 'pcc']

# run_cue_tau_1.sh / _2.sh의 TEMPERATURES와 동일하게 맞춘다.
TEMPERATURES = [0.25, 0.5, 1.0, 2.0, 4.0]

model_sources = {f'TAU={temp}': os.path.join(cue_base, '{dataset}', '{target}', 'v1', f'TAU{temp}') for temp in TEMPERATURES}

# rmse는 낮을수록, r2/pcc는 높을수록 좋은 성능
LOWER_IS_BETTER = {'rmse': True, 'r2': False, 'pcc': False}

In [2]:
def load_metric(model_dir_template, dataset, target):
    """cmfe의 metrics.csv(train.py::aggregate_metrics 출력)를 읽어
    model_name == 'mlp' 행의 rmse/r2/pcc를 뽑아온다."""
    metrics_path = os.path.join(model_dir_template.format(dataset=dataset, target=target), 'metrics.csv')
    if not os.path.exists(metrics_path):
        return None

    df_m = pd.read_csv(metrics_path)
    mlp_row = df_m[df_m['model_name'] == 'mlp']
    if mlp_row.empty:
        return None

    values = {}
    for col in mlp_row.columns:
        if col in ['model_name', 'Unnamed: 0']:
            continue
        norm_col = col.lower()
        if norm_col in metrics_of_interest:
            values[norm_col] = mlp_row[col].values[0]
    return values if values else None


raw_rows = []
for d_name in datasets:
    for target in targets:
        for model_name, dir_template in model_sources.items():
            vals = load_metric(dir_template, d_name, target)
            if vals is not None:
                raw_rows.append({'Dataset': d_name, 'Target': target, 'Model': model_name, **vals})

df_total = pd.DataFrame(raw_rows)
print(f"✅ 데이터 로드 완료: {len(df_total)}개의 항목이 수집되었습니다 (기대값: 2 x 4 x 5 = 40).")

✅ 데이터 로드 완료: 40개의 항목이 수집되었습니다 (기대값: 2 x 4 x 5 = 40).


In [3]:
def plot_tau_line(df_all, dataset, target_name, metric_name, output_dir):
    """temperature(x축, log2 스케일) 대비 metric 값을 line plot으로 표시한다."""
    df_sub = df_all[(df_all['Dataset'] == dataset) & (df_all['Target'] == target_name)]
    if df_sub.empty or metric_name not in df_sub.columns:
        return

    xs, ys = [], []
    for temp in TEMPERATURES:
        row = df_sub[df_sub['Model'] == f'TAU={temp}']
        if not row.empty:
            xs.append(temp)
            ys.append(row[metric_name].values[0])

    if not xs:
        return

    fig, ax = plt.subplots(figsize=(6, 4))
    ax.plot(xs, ys, marker='o', color='#118ab2', linewidth=2, label='TAU sweep')

    ax.set_xscale('log', base=2)
    ax.set_xticks(TEMPERATURES)
    ax.set_xticklabels([str(t) for t in TEMPERATURES])

    ax.set_xlabel('Temperature (tau)')
    ax.set_ylabel(metric_name.upper())
    ax.set_title(f"{dataset} / {target_name} / {metric_name}")
    ax.grid(True, linestyle='--', alpha=0.5)

    plt.tight_layout()
    os.makedirs(output_dir, exist_ok=True)
    save_path = os.path.join(output_dir, f"{target_name}_{metric_name}.png")
    plt.savefig(save_path, dpi=300)
    plt.close()
    print(f"✅ 저장 완료: {save_path}")

In [4]:
plot_base_dir = os.path.join(cue_base, 'plots', 'tau_comparison')

for d_name in datasets:
    for target in targets:
        g_name = 'DIS' if d_name == 'DIS' else 'INH'
        output_path = os.path.join(plot_base_dir, g_name)
        for m in metrics_of_interest:
            plot_tau_line(df_total, d_name, target, m, output_path)

✅ 저장 완료: /data/rlawlsgurjh/work/gitlab/CUE/cmfe/results/plots/tau_comparison/DIS/PI_rmse.png


✅ 저장 완료: /data/rlawlsgurjh/work/gitlab/CUE/cmfe/results/plots/tau_comparison/DIS/PI_r2.png
✅ 저장 완료: /data/rlawlsgurjh/work/gitlab/CUE/cmfe/results/plots/tau_comparison/DIS/PI_pcc.png


✅ 저장 완료: /data/rlawlsgurjh/work/gitlab/CUE/cmfe/results/plots/tau_comparison/DIS/Ssel_rmse.png


✅ 저장 완료: /data/rlawlsgurjh/work/gitlab/CUE/cmfe/results/plots/tau_comparison/DIS/Ssel_r2.png


✅ 저장 완료: /data/rlawlsgurjh/work/gitlab/CUE/cmfe/results/plots/tau_comparison/DIS/Ssel_pcc.png


✅ 저장 완료: /data/rlawlsgurjh/work/gitlab/CUE/cmfe/results/plots/tau_comparison/DIS/WS(alpha=2)_rmse.png


... (11 outputs left out)

✅ 저장 완료: /data/rlawlsgurjh/work/gitlab/CUE/cmfe/results/plots/tau_comparison/INH/WS(alpha=2)_r2.png


✅ 저장 완료: /data/rlawlsgurjh/work/gitlab/CUE/cmfe/results/plots/tau_comparison/INH/WS(alpha=2)_pcc.png


✅ 저장 완료: /data/rlawlsgurjh/work/gitlab/CUE/cmfe/results/plots/tau_comparison/INH/RS(k=2)_rmse.png


✅ 저장 완료: /data/rlawlsgurjh/work/gitlab/CUE/cmfe/results/plots/tau_comparison/INH/RS(k=2)_r2.png
✅ 저장 완료: /data/rlawlsgurjh/work/gitlab/CUE/cmfe/results/plots/tau_comparison/INH/RS(k=2)_pcc.png
